# URL-Based Phishing Website Detection Using Machine Learning and Artificial Neural Networks
**Formative Assessment 1 (FA1) / Machine Learning Project**  
*Authors: Christy Francis (124B1B005), Saksham Joshi (124B1B021), Aadesh Patil (124B1B023)*  
*Department of Computer Engineering, Pimpri Chinchwad College of Engineering, Pune*

---

### Pipeline Structure:
1. **Import Libraries**
2. **Load Dataset**
3. **Explore Dataset (EDA)**
4. **Data Preprocessing & Visualizations**
   - Missing Values & Duplicate Records
   - Target Class Distribution & Visualization (Fig. 1)
   - HTTPS Usage Analysis & Visualization (Fig. 2)
   - Dropping Non-Predictive Text Columns
   - Target Leakage Screening & Removal (URLSimilarityIndex, TLDLegitimateProb)
   - Feature Correlation Heatmap (Fig. 3)
   - Splitting Features & Target
5. **Feature Selection (SelectKBest with Mutual Information)**
   - Computing MI Scores
   - Top 20 Feature Importance Bar Plot (Fig. 4)
   - Top 30 Selected Features
6. **Train-Test Split (Stratified 80/20)**
7. **Feature Scaling (StandardScaler fitted strictly on training data)**
8. **Build Proposed Artificial Neural Network (ANN)**
9. **Train Proposed ANN Model (with Dropout & Early Stopping)**
   - Training vs Validation Curves (Fig. 5)
10. **Evaluate Model on Held-Out Test Set**
    - Confusion Matrix (TP, TN, FP, FN with False Negative highlighted) & Heatmap (Fig. 6)
    - Performance Metrics: Accuracy, Precision, Recall, F1-Score, ROC-AUC (Table 4)
11. **Feature Selection Ablation Study (Section 13, Table 7)**
12. **Comparison with Traditional Machine Learning Models (Section 12, Table 6)**
    - Logistic Regression, Decision Tree, Random Forest, LinearSVC, XGBoost
    - Performance Comparison Bar Chart (Fig. 7)


## 1. Import Libraries


In [ ]:
import os
import sys
import time
import pickle
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress non-critical warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# XGBoost
from xgboost import XGBClassifier

# TensorFlow & Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("Libraries imported successfully!")


## 2. Load Dataset


In [ ]:
# Load PhiUSIIL Phishing URL Dataset
csv_filename = "PhiUSIIL_Phishing_URL_Dataset.csv"
df = pd.read_csv(csv_filename)

print("Dataset Loaded Successfully!")
print(f"Total Rows: {df.shape[0]:,} | Total Columns: {df.shape[1]}")
df.head()


## 3. Explore Dataset (EDA)


In [ ]:
# Check column types, memory usage, and non-null counts
df.info()


In [ ]:
# Dataset dimensions
print("Shape of dataset:", df.shape)


In [ ]:
# Statistical summary of numerical features
df.describe().T.head(15)


In [ ]:
# List all 56 columns
print(list(df.columns))


In [ ]:
# Check for missing values across all columns
print("Missing values per column:")
print(df.isnull().sum())
print("\nTotal missing values in dataset:", df.isnull().sum().sum())


## 4. Data Preprocessing & Visualizations


In [ ]:
# Drop nulls if any exist and check for duplicate records
df.dropna(inplace=True)

duplicates = df.duplicated().sum()
print("Number of duplicate rows:", duplicates)

if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print("Dropped duplicates. New shape:", df.shape)
else:
    print("No duplicates found. Dataset preserved.")


In [ ]:
# Target class distribution (1 = Legitimate, 0 = Phishing)
class_counts = df["label"].value_counts()
print(class_counts)
print(f"Legitimate (1): {class_counts[1]:,} ({class_counts[1]/len(df)*100:.2f}%)")
print(f"Phishing   (0): {class_counts[0]:,} ({class_counts[0]/len(df)*100:.2f}%)")


### Fig. 1: Target Class Distribution


In [ ]:
# Visualization of class balance
plt.figure(figsize=(7, 5))
ax = sns.countplot(x='label', data=df, palette=['#ef4444', '#10b981'])
plt.title("Distribution of Target Classes (Fig. 1)", fontsize=13, fontweight='bold')
plt.xlabel("Class (0 = Phishing, 1 = Legitimate)", fontsize=11)
plt.ylabel("Number of Samples", fontsize=11)
plt.xticks([0, 1], [f"Phishing\n({class_counts[0]:,})", f"Legitimate\n({class_counts[1]:,})"])
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='baseline', fontsize=10, xytext=(0, 5), textcoords='offset points')
plt.tight_layout()
plt.show()


### Fig. 2: HTTPS Usage Across Legitimate and Phishing URLs (Paper Fig. 9)


In [ ]:
# HTTPS Usage by Class
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="IsHTTPS", hue="label", palette=['#ef4444', '#10b981'])
plt.title("HTTPS Usage Across Classes (Fig. 2 / Paper Fig. 9)", fontsize=13, fontweight='bold')
plt.xlabel("IsHTTPS (0 = HTTP / Non-Secure, 1 = HTTPS / Secure)", fontsize=11)
plt.ylabel("Number of URLs", fontsize=11)
plt.legend(title="Class", labels=["Phishing", "Legitimate"])
plt.tight_layout()
plt.show()


### Removal of Non-Predictive Identifier Columns (Section 4.4)


In [ ]:
# Drop raw text/identifier columns that cause sample memorization
df_clean = df.drop(columns=[
    "FILENAME",
    "URL",
    "Domain",
    "Title",
    "TLD"
])
print("Dropped non-predictive columns: ['FILENAME', 'URL', 'Domain', 'Title', 'TLD']")
print("Remaining columns:", df_clean.shape[1])


### Target Leakage Screening & Removal\nChecking for features that trivially separate the classes through target leakage.


In [ ]:
# 1. Inspect URLSimilarityIndex
sim_legit = df.loc[df['label'] == 1, 'URLSimilarityIndex']
sim_phish = df.loc[df['label'] == 0, 'URLSimilarityIndex']
print(f"URLSimilarityIndex in Legitimate class: mean={sim_legit.mean():.2f}, std={sim_legit.std():.4f} (100% exact 100.0)")
print(f"URLSimilarityIndex in Phishing class:   mean={sim_phish.mean():.2f}, std={sim_phish.std():.4f}")

# 2. Inspect TLDLegitimateProb
tld_var = df.groupby('TLD')['TLDLegitimateProb'].nunique().max()
print(f"TLDLegitimateProb unique values per TLD: {tld_var} (Zero variance across all rows of same TLD)")

# Drop confirmed leaking features
leaking_features = ['URLSimilarityIndex', 'TLDLegitimateProb']
df_clean = df_clean.drop(columns=leaking_features)
print(f"Dropped leaking features: {leaking_features}")
print("Final clean predictor feature count:", df_clean.shape[1] - 1)


### Fig. 3: Feature Correlation Heatmap (Paper Fig. 10)


In [ ]:
# Correlation Heatmap of usable numerical features
plt.figure(figsize=(16, 13))
corr = df_clean.drop(columns=['label']).corr()
sns.heatmap(corr, cmap="coolwarm", center=0, cbar_kws={'shrink': 0.8})
plt.title("Correlation Heatmap of Numerical Features (Fig. 3 / Paper Fig. 10)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### Separate Features (X) and Target (y)


In [ ]:
X = df_clean.drop("label", axis=1)
y = df_clean["label"].values
feature_names = list(X.columns)

print("X shape:", X.shape, "(Input Features)")
print("y shape:", y.shape, "(Binary Labels: 1 = Legitimate, 0 = Phishing)")


## 5. Stratified Train-Test Split (Section 4.6)\n**Crucial ML methodology note**: To strictly prevent silent data leakage, train-test splitting MUST happen before fitting StandardScaler or SelectKBest!


In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train_raw.shape} | y_train: {y_train.shape}")
print(f"X_test : {X_test_raw.shape}  | y_test : {y_test.shape}")
print(f"Train class balance: Legitimate={np.mean(y_train==1)*100:.2f}%, Phishing={np.mean(y_train==0)*100:.2f}%")
print(f"Test class balance:  Legitimate={np.mean(y_test==1)*100:.2f}%, Phishing={np.mean(y_test==0)*100:.2f}%")


## 6. Feature Scaling (StandardScaler, Section 6)\nFitted ONLY on `X_train_raw` to prevent test-set leakage.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

print("Scaled training features shape:", X_train_scaled.shape)
print("Scaled testing features shape :", X_test_scaled.shape)


## 7. Feature Selection (SelectKBest with Mutual Information, Section 7)


In [ ]:
# Compute Mutual Information on scaled training data
selector = SelectKBest(score_func=lambda X, y: mutual_info_classif(X, y, random_state=42), k=30)
X_train_sel30 = selector.fit_transform(X_train_scaled, y_train)
X_test_sel30 = selector.transform(X_test_scaled)

mi_scores = selector.scores_
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': mi_scores
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)
feature_importance['Rank'] = feature_importance.index + 1

print("Top 20 Features Ranked by Mutual Information:")
print(feature_importance.head(20).to_string(index=False))


### Fig. 4: Top 20 Important Features Bar Plot (Paper Fig. 8)


In [ ]:
top20 = feature_importance.head(20).iloc[::-1]

plt.figure(figsize=(10, 8))
plt.barh(top20["Feature"], top20["Importance"], color='#2563eb', edgecolor='black', alpha=0.85)
plt.xlabel("Mutual Information Score", fontsize=11, fontweight='bold')
plt.ylabel("Feature Name", fontsize=11, fontweight='bold')
plt.title("Top 20 Important Features for Phishing Detection (Fig. 4 / Paper Fig. 8)", fontsize=13, fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


## 8. Build Proposed Artificial Neural Network (ANN, Section 8)\nArchitecture matching Table 3 of the paper:  \n`Input(30) -> Dense(128, relu) -> Dropout(0.3) -> Dense(64, relu) -> Dropout(0.3) -> Dense(32, relu) -> Dense(1, sigmoid)`


In [ ]:
# Carve 10% stratified validation split out of training set
X_tr_ann, X_val_ann, y_tr_ann, y_val_ann = train_test_split(
    X_train_sel30,
    y_train,
    test_size=0.10,
    stratify=y_train,
    random_state=42
)

ann_model = Sequential([
    Input(shape=(30,)),
    Dense(128, activation="relu", name="Dense_1"),
    Dropout(0.30, name="Dropout_1"),
    Dense(64, activation="relu", name="Dense_2"),
    Dropout(0.30, name="Dropout_2"),
    Dense(32, activation="relu", name="Dense_3"),
    Dense(1, activation="sigmoid", name="Output_Sigmoid")
], name="Phishing_ANN_Proposed")

ann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann_model.summary()


## 9. Train Proposed ANN Model\nUsing Adam optimizer, binary cross-entropy, batch size 512, and early stopping.


In [ ]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history = ann_model.fit(
    X_tr_ann,
    y_tr_ann,
    validation_data=(X_val_ann, y_val_ann),
    epochs=40,
    batch_size=512,
    callbacks=[early_stopping],
    verbose=1
)


### Fig. 5: ANN Training vs Validation Curves (Paper Fig. 11)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy Curve
ax1.plot(history.history['accuracy'], label='Train Accuracy', color='#2563eb', lw=2)
ax1.plot(history.history['val_accuracy'], label='Val Accuracy', color='#10b981', lw=2, linestyle='--')
ax1.set_title("ANN Training vs Validation Accuracy (Fig. 5 / Paper Fig. 11)", fontsize=12, fontweight='bold')
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss Curve
ax2.plot(history.history['loss'], label='Train Loss', color='#ef4444', lw=2)
ax2.plot(history.history['val_loss'], label='Val Loss', color='#f59e0b', lw=2, linestyle='--')
ax2.set_title("ANN Training vs Validation Loss (Binary Cross-Entropy)", fontsize=12, fontweight='bold')
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 10. Evaluate Proposed ANN on Held-Out Test Set (Section 10 & 11)\n**Per Section 10 of Paper**: Phishing is treated as the Positive Class.


In [ ]:
y_prob_legit = ann_model.predict(X_test_sel30, batch_size=512).ravel()
y_prob_phish = 1.0 - y_prob_legit
y_pred_phish = (y_prob_phish >= 0.5).astype(int)
y_test_phish = (y_test == 0).astype(int)

# Metrics
acc = accuracy_score(y_test_phish, y_pred_phish)
prec = precision_score(y_test_phish, y_pred_phish)
rec = recall_score(y_test_phish, y_pred_phish)
f1 = f1_score(y_test_phish, y_pred_phish)
auc = roc_auc_score(y_test_phish, y_prob_phish)
cm = confusion_matrix(y_test_phish, y_pred_phish)
tn, fp, fn, tp = cm.ravel()

print("=" * 60)
print("Table 4: Performance of Proposed ANN (Held-Out Test Set):")
print("=" * 60)
print(f"Accuracy:  {acc*100:.3f}%")
print(f"Precision: {prec*100:.3f}%")
print(f"Recall:    {rec*100:.3f}%")
print(f"F1-Score:  {f1*100:.3f}%")
print(f"ROC-AUC:   {auc:.5f}")
print("-" * 60)
print(f"TP (Phishing correctly caught):     {tp:,}")
print(f"TN (Legitimate correctly cleared):  {tn:,}")
print(f"FP (Legitimate falsely blocked):    {fp:,}")
print(f"FN (Phishing MISSED - Costliest!):  {fn:,}")
print("=" * 60)


### Fig. 6: Confusion Matrix Heatmap


In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap([[tn, fp], [fn, tp]], annot=True, fmt=',d', cmap="Blues",
            xticklabels=["Legitimate", "Phishing"],
            yticklabels=["Legitimate", "Phishing"])
plt.title("Confusion Matrix - Proposed ANN\n(Phishing = Positive Class)", fontsize=13, fontweight='bold')
plt.xlabel("Predicted Class", fontsize=11, fontweight='bold')
plt.ylabel("True Class", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


## 11. Feature Selection Ablation Study (Section 13, Table 7)\nComparing ANN with Mutual Information (30 features) vs ANN without feature selection (All 48 usable features).


In [ ]:
X_tr_ab, X_val_ab, y_tr_ab, y_val_ab = train_test_split(
    X_train_scaled,
    y_train,
    test_size=0.10,
    stratify=y_train,
    random_state=42
)

ablation_model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(128, activation="relu"),
    Dropout(0.30),
    Dense(64, activation="relu"),
    Dropout(0.30),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

ablation_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
ablation_model.fit(
    X_tr_ab, y_tr_ab,
    validation_data=(X_val_ab, y_val_ab),
    epochs=40, batch_size=512,
    callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)],
    verbose=0
)

# Evaluate Ablation Model
y_prob_ab_phish = 1.0 - ablation_model.predict(X_test_scaled, batch_size=512).ravel()
y_pred_ab_phish = (y_prob_ab_phish >= 0.5).astype(int)

ablation_df = pd.DataFrame([
    {
        "Configuration": "ANN without feature selection",
        "Features": X.shape[1],
        "Accuracy": accuracy_score(y_test_phish, y_pred_ab_phish),
        "Precision": precision_score(y_test_phish, y_pred_ab_phish),
        "Recall": recall_score(y_test_phish, y_pred_ab_phish),
        "F1-Score": f1_score(y_test_phish, y_pred_ab_phish),
        "ROC-AUC": roc_auc_score(y_test_phish, y_prob_ab_phish)
    },
    {
        "Configuration": "ANN with Mutual Information",
        "Features": 30,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": auc
    }
])

print("Table 7: Feature Selection Ablation Study Results:")
print(ablation_df.to_string(index=False))


## 12. Comparison with Traditional Machine Learning Models (Section 12, Table 6)\nComparing Proposed ANN with Logistic Regression, Decision Tree, Random Forest, LinearSVC, and XGBoost on the identical 30 features.


In [ ]:
models = [
    ("Logistic Regression", LogisticRegression(max_iter=1000, random_state=42)),
    ("Decision Tree", DecisionTreeClassifier(random_state=42)),
    ("Random Forest", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ("Support Vector Machine (LinearSVC)", LinearSVC(dual='auto', random_state=42)),
    ("XGBoost", XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=42, n_jobs=-1))
]

comp_results = []
for name, clf in models:
    clf.fit(X_train_sel30, y_train)
    y_pred = clf.predict(X_test_sel30)
    
    if hasattr(clf, "predict_proba"):
        p_phish = clf.predict_proba(X_test_sel30)[:, 0]
    elif hasattr(clf, "decision_function"):
        p_phish = -clf.decision_function(X_test_sel30)
    else:
        p_phish = (y_pred == 0).astype(float)
        
    p_pred_phish = (y_pred == 0).astype(int)
    cm_i = confusion_matrix(y_test_phish, p_pred_phish)
    tn_i, fp_i, fn_i, tp_i = cm_i.ravel()
    
    comp_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_phish, p_pred_phish),
        "Precision": precision_score(y_test_phish, p_pred_phish),
        "Recall": recall_score(y_test_phish, p_pred_phish),
        "F1-Score": f1_score(y_test_phish, p_pred_phish),
        "ROC-AUC": roc_auc_score(y_test_phish, p_phish),
        "TP": tp_i, "TN": tn_i, "FP": fp_i, "FN": fn_i
    })

# Add ANN
comp_results.append({
    "Model": "Proposed ANN (30 Features)",
    "Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1, "ROC-AUC": auc,
    "TP": tp, "TN": tn, "FP": fp, "FN": fn
})

comp_df = pd.DataFrame(comp_results)
print("Table 6: Consolidated Model Comparison (Held-Out Test Set):")
print(comp_df.to_string(index=False))


### Fig. 7: Model Comparison Bar Chart


In [ ]:
models_list = comp_df["Model"].tolist()
acc_list = [v * 100 for v in comp_df["Accuracy"].tolist()]
f1_list = [v * 100 for v in comp_df["F1-Score"].tolist()]

x_idx = np.arange(len(models_list))
width = 0.35

plt.figure(figsize=(12, 6))
plt.bar(x_idx - width/2, acc_list, width, label='Accuracy (%)', color='#2563eb', edgecolor='black', alpha=0.9)
plt.bar(x_idx + width/2, f1_list, width, label='F1-Score (%)', color='#10b981', edgecolor='black', alpha=0.9)
plt.ylabel('Score (%)', fontsize=11, fontweight='bold')
plt.title('Model Performance Comparison on 30 Selected Features (Fig. 7)', fontsize=13, fontweight='bold')
plt.xticks(x_idx, [m.replace("Support Vector Machine (LinearSVC)", "LinearSVC").replace("Proposed ANN (30 Features)", "Proposed ANN") for m in models_list], rotation=15, fontsize=10, fontweight='bold')
plt.ylim([99.5, 100.05])
plt.legend(loc='lower right', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
